<a href="https://colab.research.google.com/github/isuru-phi6/ionospheric_plasma_bubbles/blob/main/Variable_Screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Variable identification and screening
**Run this notebook before the Objective 1 notebook.**

In any statistical study, before building models we:
1. **identify the dependent variable** (what we want to explain) and the **independent variables** (what might explain it)
2. decide the **type** of each variable, because the type decides which test to use
3. **describe** the data: summaries, missing values, distributions
4. **screen** each independent variable: is it related to the dependent variable, and how strongly?
5. check whether the independent variables are **related to each other** (multicollinearity)
6. write down the decisions for the Methods section

**Data:** `passes_night.csv` from the main notebook (Step 7): one row per Swarm night pass over 70–90°E, ±20° magnetic latitude, 18–06 LT.

**Outputs:** tables go to `results/O0/` and figures to `figures/O0/` in your Drive folder.

---
## 1. Load the data

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/EPB_SriLanka")
except ImportError:
    BASE_DIR = Path(os.environ.get("EPB_DIR", "EPB_SriLanka"))

DATA_FILE = BASE_DIR / "results" / "passes_night.csv"
FIG_DIR = BASE_DIR / "figures" / "O0"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = BASE_DIR / "results" / "O0"; TAB_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name): plt.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

p = pd.read_csv(DATA_FILE, parse_dates=["t_start", "t_end", "t_eq"])
print(f"Rows (passes): {len(p):,}   Columns: {p.shape[1]}")
print("Columns in the file:", list(p.columns))

---
## 2. Identify the dependent and independent variables

### Dependent variable (Y)
**`bubble`**: 1 if Swarm saw a plasma bubble during the pass, 0 if not.
- It is **binary** (two categories). So the right regression model is **logistic regression**, not linear regression.
- Its summary is a **proportion**: the occurrence rate.

### Independent variables (X)
These were chosen **in advance from physics theory and earlier studies** (theory-driven selection), and are then checked with the data in this notebook.

| Column | Meaning | Type | Physical reason |
|---|---|---|---|
| `LT` | Local time of the pass (hours) | Continuous, cyclic | Pre-reversal enhancement (PRE) timing after sunset |
| `doy` | Day of year | Continuous, cyclic | Season: sunset alignment along magnetic field lines |
| `F107_81` | Solar flux, 81-day mean (sfu) | Continuous | Solar EUV → ionosphere density and PRE strength |
| `F107` | Solar flux, daily (sfu) | Continuous | Short-term solar EUV changes |
| `Kp` | Planetary geomagnetic index (0–9) | Continuous (ordinal, treated as continuous) | Geomagnetic activity |
| `Dst` | Ring-current storm index now (nT) | Continuous | Storm intensity |
| `Dst_min24` | Lowest Dst in previous 24 h (nT) | Continuous | Strength of a recent storm |
| `dDst_3h` | Change in Dst over 3 h (nT) | Continuous | Storm developing (negative) or recovering |
| `AE`, `AE_max6h` | Auroral index now / max in 6 h (nT) | Continuous | High-latitude heating → disturbance dynamo |
| `Bz_lag1`, `Bz_mean3h` | IMF Bz 1 h before / mean of 3 h (nT) | Continuous | Southward Bz → prompt penetration electric field |
| `V_lag1` | Solar wind speed 1 h before (km/s) | Continuous | Solar wind driving |
| `alt_km` | Satellite altitude (km) | Continuous | Detection effect |
| `season` | Lloyd season (from month) | Categorical, 4 levels | Grouped version of `doy` |
| `LT2` | 2-hour local-time bin | Categorical (ordinal), 6 levels | Grouped version of `LT` |
| `sat` | Swarm A, B or C | Categorical, 3 levels | Satellite differences |
| `year` | Year | Categorical, 13 levels | Solar cycle |

In [ ]:
Y = "bubble"
CONTINUOUS = {
    "LT_from18": "Hours after 18 LT", "F107_81": "Solar flux, 81-day (sfu)", "F107": "Solar flux, daily (sfu)",
    "Kp": "Kp index", "Dst": "Dst now (nT)", "Dst_min24": "Minimum Dst, previous 24 h (nT)",
    "dDst_3h": "Dst change over 3 h (nT)", "AE": "AE now (nT)", "AE_max6h": "Maximum AE, previous 6 h (nT)",
    "Bz_lag1": "IMF Bz, 1 h before (nT)", "Bz_mean3h": "Mean IMF Bz, previous 3 h (nT)",
    "V_lag1": "Solar wind speed, 1 h before (km/s)", "alt_km": "Satellite altitude (km)",
}
# categorical versions
season_map = {1: "Dec solstice", 2: "Dec solstice", 11: "Dec solstice", 12: "Dec solstice",
              3: "Mar equinox", 4: "Mar equinox", 9: "Sep equinox", 10: "Sep equinox",
              5: "Jun solstice", 6: "Jun solstice", 7: "Jun solstice", 8: "Jun solstice"}
p["season"] = pd.Categorical(p["month"].map(season_map),
                             categories=["Mar equinox", "Jun solstice", "Sep equinox", "Dec solstice"])
lt2 = ["18-20", "20-22", "22-24", "00-02", "02-04", "04-06"]
p["LT2"] = pd.Categorical(np.array(lt2)[(p["LT_from18"] // 2).astype(int).clip(0, 5)], categories=lt2, ordered=True)
CATEGORICAL = {"season": "Season (Lloyd)", "LT2": "Local time, 2-h bins", "sat": "Satellite", "year": "Year"}

print("Dependent variable:", Y, "| values:", sorted(p[Y].unique()))
print("Continuous independent variables:", len(CONTINUOUS), "| Categorical:", len(CATEGORICAL))

**Note on `LT_from18`:** local time is cyclic (23 LT is next to 00 LT). For screening, we use **hours after 18 LT** (0 = 18 LT, 6 = midnight, 12 = 06 LT), which runs in a straight line through one night. The models use sin/cos or spline forms instead.

---
## 3. Describe the dependent variable

In [ ]:
counts = p[Y].value_counts().rename({0: "no bubble (0)", 1: "bubble (1)"})
rate = p[Y].mean()
lo, hi = stats.binomtest(int(p[Y].sum()), len(p)).proportion_ci(method="wilson")
print(counts.to_string())
print(f"\nOccurrence rate = {rate:.3f} ({100*rate:.1f}%), 95% Wilson CI {100*lo:.1f}-{100*hi:.1f}%")
print("(This interval assumes independent passes; Objective 1 gives the cluster-bootstrap interval, which is slightly wider.)")
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.bar(["no bubble", "bubble"], counts.values, color=["lightgrey", "tab:red"])
for i, v in enumerate(counts.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_ylabel("Number of passes"); ax.set_title("Dependent variable: bubble")
savefig("O0_dependent_variable"); plt.show()

**Interpretation.** Only about 6% of passes have a bubble, so the outcome is **imbalanced**:
- With rare events, accuracy is a poor measure of model quality (Objective 2 uses better scores).
- Each category of a categorical variable needs enough bubble passes for a stable estimate.

---
## 4. Describe the independent variables
### 4a. Summary statistics and missing values

In [ ]:
desc = p[list(CONTINUOUS)].describe(percentiles=[0.25, 0.5, 0.75]).T
desc["missing"] = p[list(CONTINUOUS)].isna().sum()
desc["missing %"] = 100 * desc["missing"] / len(p)
desc.index = [CONTINUOUS[c] for c in desc.index]
desc = desc[["count", "missing", "missing %", "mean", "std", "min", "25%", "50%", "75%", "max"]]
display(desc.round(2)); desc.to_csv(TAB_DIR / "O0_descriptive_continuous.csv")
for c, name in CATEGORICAL.items():
    print(f"\n{name}:"); print(p[c].value_counts(sort=False).to_string())

**What to check:**
- **Missing values:** a few percent or less is fine. Large gaps would need a plan, such as imputation or dropping the variable.
- **Min and max:** look for impossible values, for example Kp > 9 or negative solar wind speed.
- **Mean far from median:** this indicates a **skewed** distribution. For example, AE has a long right tail because a few very disturbed hours have huge values. Skewness is one reason to use **non-parametric tests** (Mann–Whitney) instead of the t-test.

### 4b. Distributions
Histograms show shape (symmetric or skewed) and outliers.

In [ ]:
cols = list(CONTINUOUS)
fig, axes = plt.subplots(4, 4, figsize=(17, 13)); axes = axes.ravel()
for ax, c in zip(axes, cols):
    ax.hist(p[c].dropna(), bins=40, color="steelblue")
    ax.set_title(CONTINUOUS[c], fontsize=9)
    sk = stats.skew(p[c].dropna()); ax.text(0.97, 0.9, f"skew={sk:.1f}", transform=ax.transAxes, ha="right", fontsize=8)
for ax in axes[len(cols):]: ax.axis("off")
plt.tight_layout(); savefig("O0_histograms"); plt.show()

**Skewness rule of thumb:** |skew| < 0.5 means roughly symmetric, 0.5–1 moderately skewed, > 1 strongly skewed. Strongly skewed variables (AE, Dst) confirm the choice of non-parametric tests.

---
## 5. Screening: is each independent variable related to `bubble`?

### 5a. Visual comparison (boxplots)
For each continuous variable, compare its distribution in **bubble** vs **no-bubble** passes. If the boxes are clearly shifted, the variable is related to bubble occurrence.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(17, 13)); axes = axes.ravel()
for ax, c in zip(axes, cols):
    d0 = p.loc[p[Y] == 0, c].dropna(); d1 = p.loc[p[Y] == 1, c].dropna()
    ax.boxplot([d0, d1], showfliers=False, widths=0.6)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["no bubble", "bubble"])
    ax.set_title(CONTINUOUS[c], fontsize=9)
for ax in axes[len(cols):]: ax.axis("off")
plt.tight_layout(); savefig("O0_boxplots_by_bubble"); plt.show()

### 5b. Statistical tests for continuous variables
For each continuous variable we compute:

| Measure | What it does | Why this one |
|---|---|---|
| **Median in each group** | Typical value for no-bubble and bubble passes | Robust to skewness |
| **Mann–Whitney U test** | Tests whether values tend to be higher in one group | Non-parametric: no normality assumption, suits skewed data |
| **AUC** (= Mann–Whitney U ÷ (n₀·n₁)) | Probability that a random bubble pass has a higher value than a random no-bubble pass. 0.5 = no relationship | An **effect size**: shows *how strong*, not only *whether* |
| **Point-biserial correlation** | Pearson correlation between a 0/1 variable and a continuous one | Familiar correlation scale (−1 to 1) |
| **Univariate logistic regression** | Odds ratio for a **1 standard deviation** increase | Same model family as the main analysis; ORs per SD are comparable across variables |

**Multiple testing.** We run 13 tests, so some small p-values could appear by chance. We apply the **Holm correction**, which controls the chance of any false positive across all tests.

**Interpreting AUC as effect size** (common guideline): 0.50–0.56 negligible, 0.56–0.64 small, 0.64–0.71 medium, > 0.71 large.

In [ ]:
def auc_label(a):
    a = max(a, 1 - a)
    return "negligible" if a < 0.56 else "small" if a < 0.64 else "medium" if a < 0.71 else "large"

rows = []
for c, name in CONTINUOUS.items():
    d = p[[c, Y]].dropna()
    x0, x1 = d.loc[d[Y] == 0, c], d.loc[d[Y] == 1, c]
    u = stats.mannwhitneyu(x1, x0, alternative="two-sided")
    auc = roc_auc_score(d[Y], d[c])
    r_pb, p_pb = stats.pointbiserialr(d[Y], d[c])
    z = (d[c] - d[c].mean()) / d[c].std()
    lr = sm.Logit(d[Y], sm.add_constant(z)).fit(disp=0)
    or_sd = np.exp(lr.params.iloc[1]); ci = np.exp(lr.conf_int().iloc[1])
    rows.append({"variable": name, "column": c, "n": len(d),
                 "median (no bubble)": x0.median(), "median (bubble)": x1.median(),
                 "Mann-Whitney U": u.statistic, "p (MW)": u.pvalue,
                 "AUC": auc, "effect size": auc_label(auc),
                 "direction": "higher in bubble passes" if auc > 0.5 else "lower in bubble passes",
                 "point-biserial r": r_pb,
                 "OR per 1 SD": or_sd, "OR CI low": ci.iloc[0], "OR CI high": ci.iloc[1]})
cont_tab = pd.DataFrame(rows)
cont_tab["p (Holm)"] = multipletests(cont_tab["p (MW)"], method="holm")[1]
cont_tab["strength"] = cont_tab["AUC"].apply(lambda a: max(a, 1 - a))
cont_tab = cont_tab.sort_values("strength", ascending=False).drop(columns="strength")
cont_tab.to_csv(TAB_DIR / "O0_screening_continuous.csv", index=False)
show = cont_tab[["variable", "median (no bubble)", "median (bubble)", "AUC", "effect size", "direction",
                 "point-biserial r", "OR per 1 SD", "OR CI low", "OR CI high", "p (Holm)"]].copy()
show["p (Holm)"] = show["p (Holm)"].apply(lambda v: f"{v:.1e}")
display(show.round(3))

**How to read the table.**
- **Direction:** for example, solar flux is *higher* in bubble passes, and Dst is *lower* (more negative, meaning stronger storms) in bubble passes.
- **OR per 1 SD:** above 1 means the odds of a bubble increase as the variable increases; below 1 means they decrease. For example, OR = 1.9 means one standard deviation more of that variable gives about 1.9 times the odds.
- **p (Holm):** the p-value after correcting for 13 tests.
- **Effect size matters more than p here.** With more than 11,000 passes, even tiny effects give very small p-values. A variable can be "significant" yet have a *negligible* AUC. That's why we report both.

### 5c. Statistical tests for categorical variables
For each categorical variable we compute:
- the **occurrence rate in each category**, with a 95% Wilson CI
- the **chi-square test of independence**: is occurrence the same in every category?
- **Cramér's V** as the effect size (0 = no association, 1 = perfect). Guideline for these table sizes: about 0.1 small, 0.3 medium, 0.5 large.

The chi-square test needs **expected counts ≥ 5** in every cell, and the code checks this.

In [ ]:
rows, rate_tables = [], {}
for c, name in CATEGORICAL.items():
    ct = pd.crosstab(p[c], p[Y])
    chi2, pv, dof, expected = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    rows.append({"variable": name, "levels": ct.shape[0], "chi-square": chi2, "df": dof, "p": pv,
                 "Cramer's V": v, "min expected count": expected.min()})
    k, n = ct[1], ct.sum(1)
    lo_, hi_ = zip(*[stats.binomtest(int(ki), int(ni)).proportion_ci(method="wilson") for ki, ni in zip(k, n)])
    rate_tables[c] = pd.DataFrame({"passes": n, "bubbles": k, "rate %": 100 * k / n,
                                   "CI low %": 100 * np.array(lo_), "CI high %": 100 * np.array(hi_)})
cat_tab = pd.DataFrame(rows)
cat_tab["p (Holm)"] = multipletests(cat_tab["p"], method="holm")[1]
cat_tab.to_csv(TAB_DIR / "O0_screening_categorical.csv", index=False)
display(cat_tab.round(4))
for c, t in rate_tables.items():
    print(f"\n{CATEGORICAL[c]}"); display(t.round(2))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for ax, (c, t) in zip(axes, rate_tables.items()):
    x = range(len(t))
    ax.errorbar(x, t["rate %"], yerr=[t["rate %"] - t["CI low %"], t["CI high %"] - t["rate %"]], fmt="o", capsize=3)
    ax.set_xticks(list(x)); ax.set_xticklabels([str(i) for i in t.index], rotation=45, fontsize=8)
    ax.set_title(CATEGORICAL[c]); ax.set_ylabel("Occurrence rate (%)")
plt.tight_layout(); savefig("O0_rates_categorical"); plt.show()

**How to read this.** A large chi-square with a small p means occurrence differs between categories. Cramér's V says how strongly. If a variable such as **satellite** shows no association on its own, that's still worth knowing; it can still be kept in the multivariable model as a control.

---
## 6. Are the independent variables related to each other? (multicollinearity)
If two independent variables carry almost the same information, a regression model can't separate their effects. Standard errors grow, and coefficients can even change sign. We check this in two ways:
1. **Spearman correlation matrix**: |ρ| > 0.7 means a strong relationship.
2. **Variance inflation factor (VIF)**: how much a coefficient's variance is inflated by the other variables. Guideline: VIF < 5 fine, 5–10 moderate, > 10 serious.

In [ ]:
corr = p[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8); ax.grid(False)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
plt.colorbar(im, label="Spearman correlation"); ax.set_title("Correlation between independent variables")
plt.tight_layout(); savefig("O0_correlation_matrix"); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().rename("rho").reset_index())
pairs = pairs.reindex(pairs["rho"].abs().sort_values(ascending=False).index)
print("Strongly related pairs (|rho| > 0.7):"); display(pairs[pairs["rho"].abs() > 0.7].round(2))

In [ ]:
def vif_table(columns):
    X = sm.add_constant(p[columns].dropna())
    return pd.Series({c: variance_inflation_factor(X.values, i) for i, c in enumerate(X.columns) if c != "const"}, name="VIF")

vif_all = vif_table([c for c in cols if c != "LT_from18"])
# the reduced set used in the Objective 1 model: one variable per physical driver
reduced = ["F107_81", "Kp", "Dst_min24", "Bz_mean3h", "V_lag1", "alt_km", "LT_from18"]
vif_red = vif_table(reduced)
vif = pd.concat([vif_all.rename("VIF, all variables"), vif_red.rename("VIF, one per driver")], axis=1)
vif["level (all variables)"] = vif["VIF, all variables"].apply(lambda v: "" if pd.isna(v) else "fine" if v < 5 else "moderate" if v < 10 else "serious")
vif.to_csv(TAB_DIR / "O0_vif.csv"); display(vif.round(2))

**How to read this.**
- The **two F10.7 measures**, the **two Dst measures**, and **Kp with AE** are strongly related, because they measure the same physical thing.
- With **one variable per driver**, all VIFs are low, so the Objective 1 regression is safe.
- The machine-learning models in Objectives 2 and 3 can use all variables, because trees are not harmed by collinearity in their predictions. Their *interpretation*, however, should treat correlated variables together. That's why Objective 3 reports storm indices as one "storm activity" group.

---
## 7. Confounding check: can a bivariate result be misleading?
A variable can look related to `bubble` only because it is related to another variable that truly matters. That is **confounding**. Example: **satellite altitude** vs **solar flux**. If altitude changed over the years, and solar flux also changed over the years, altitude can look important when it isn't.

Below we compare the odds ratio of altitude **alone** with the odds ratio **adjusted** for solar flux, local time and season.

In [ ]:
d = p[[Y, "alt_km", "F107_81", "LT_from18", "month"]].dropna().copy()
d["alt_10km"] = d["alt_km"] / 10
crude = sm.Logit(d[Y], sm.add_constant(d[["alt_10km"]])).fit(disp=0)
X_adj = pd.get_dummies(d[["alt_10km", "F107_81", "LT_from18", "month"]].astype({"month": str}), columns=["month"], drop_first=True).astype(float)
X_adj["LT_sq"] = d["LT_from18"] ** 2
adj = sm.Logit(d[Y], sm.add_constant(X_adj)).fit(disp=0)
conf_tab = pd.DataFrame({
    "OR per 10 km": [np.exp(crude.params["alt_10km"]), np.exp(adj.params["alt_10km"])],
    "CI low": [np.exp(crude.conf_int().loc["alt_10km", 0]), np.exp(adj.conf_int().loc["alt_10km", 0])],
    "CI high": [np.exp(crude.conf_int().loc["alt_10km", 1]), np.exp(adj.conf_int().loc["alt_10km", 1])],
}, index=["Crude (altitude alone)", "Adjusted (solar flux, local time, month)"])
display(conf_tab.round(3))
print("Correlation of altitude with F10.7 (Spearman):", round(p[["alt_km", "F107_81"]].corr(method="spearman").iloc[0, 1], 2))

If the crude and adjusted odds ratios differ a lot, or even point in opposite directions, the bivariate result was **confounded**. This is why the screening results are only a first look, and the **adjusted** results from the multivariable model in Objective 1 are the ones to report.

---
## 8. Summary table and decisions

In [ ]:
summary = cont_tab[["variable", "AUC", "effect size", "direction", "OR per 1 SD", "p (Holm)"]].copy()
summary["type"] = "continuous"
cat_sum = cat_tab[["variable", "Cramer's V", "p (Holm)"]].copy(); cat_sum["type"] = "categorical"
cat_sum["effect size"] = cat_sum["Cramer's V"].apply(lambda v: "negligible" if v < 0.05 else "small" if v < 0.2 else "medium" if v < 0.4 else "large")
summary = pd.concat([summary, cat_sum], ignore_index=True)
summary["related to bubble? (p<0.05 after Holm)"] = np.where(summary["p (Holm)"] < 0.05, "yes", "no")
summary["p (Holm)"] = summary["p (Holm)"].apply(lambda v: f"{v:.1e}")
summary.to_csv(TAB_DIR / "O0_screening_summary.csv", index=False)
display(summary.round(3))

strong = cont_tab[cont_tab["AUC"].apply(lambda a: max(a, 1 - a)) >= 0.64]["variable"].tolist()
none = summary[summary["related to bubble? (p<0.05 after Holm)"] == "no"]["variable"].tolist()
print("\nVariables with at least a medium effect:", strong)
print("Variables NOT related to bubble on their own:", none if none else "none")

### Decisions carried into the analysis
1. **Dependent variable:** `bubble` (binary), so use **logistic regression** for statistical modelling and **probability models** for machine learning.
2. **Independent variables:** all physically motivated variables are kept. Screening confirms that almost all are related to `bubble`. Variables are **not** dropped by p-value alone: with a large sample everything is significant, and a variable that looks unrelated on its own can matter after adjustment (and the reverse).
3. **Collinearity:** in regression models (Objective 1), use **one variable per physical driver** (81-day F10.7, Kp, Dst minimum 24 h, Bz, solar wind speed). In machine learning (Objectives 2–3), all variables can be used, but correlated variables are interpreted **as groups**.
4. **Non-linear and cyclic effects:** local time and season are cyclic, so use bins, sin/cos terms or splines, never a single straight line.
5. **Confounding:** report **adjusted** effects from the multivariable model, not the bivariate ones.

### Text for the Methods section
> The dependent variable was the binary occurrence of an EPB on each Swarm night pass. Candidate predictors were selected a priori from established EPB physics: local time, season, solar EUV flux, geomagnetic and auroral activity, interplanetary magnetic field and solar-wind conditions, and satellite altitude. Their bivariate association with EPB occurrence was assessed with Mann–Whitney U tests for continuous variables and chi-square tests of independence for categorical variables, with Holm correction for multiple testing. Effect sizes were reported as the area under the ROC curve (AUC) and Cramér's V, respectively. Multicollinearity was assessed using Spearman correlations and variance inflation factors. Collinear predictors describing the same physical driver were represented by a single variable in regression models.